<a href="https://colab.research.google.com/github/Renelle24/FUNDAI-Laboratories-Alemios/blob/main/Lab4_Logic_KR_Alemios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

** Name :** Renelle Alemios

** Course :** BSIT - 4

** Section :** AI-ELFUNDAI
** Date :** 09/15/26

** GitHub URL :** https://github.com/Renelle24/FUNDAI-Laboratories-Alemios.git

## Description
This laboratory uses Python and SymPy to perform truth table generation,
satisfiability checking, theorem proving, and logical deduction.

In [1]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

P, Q = symbols('P Q')

def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [2]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True

In [3]:
law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [4]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False

In [5]:
print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [6]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb

In [7]:
def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False

In [8]:
def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

In [9]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Result: Entailment holds.
------------------------------------------------------------


True

In [10]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [11]:
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example
by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:
- Human_Socrates -> Mortal_Socrates


In [12]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(kb_socrates, Mortal_Socrates, "Grounded FOL: Socrates is mortal")

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [13]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal

constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
------------------------------------------------------------


True